In [13]:
from google.colab import drive
drive.mount("/content/drive")
dataset_folder = "/content/drive/MyDrive/datasets/dataset"
dataset_folder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/datasets/dataset'

In [14]:
# TASK 1


import os

def print_tree(folder, indent=""):
    for item in os.listdir(folder):
        path = os.path.join(folder, item)
        # print(path)
        print(indent + "|-- " + item)
        for i in os.listdir(path):
            new_path = os.path.join(path, i)
            print(f"{(indent + "    |-- " + i):<22}  -->  {len(os.listdir(new_path))} ")
            

print("dataset/")
print_tree(dataset_folder)

dataset/
|-- validation
    |-- non_recyclable  -->  31 
    |-- organic         -->  205 
    |-- recyclable      -->  289 
|-- train
    |-- recyclable      -->  1104 
    |-- non_recyclable  -->  104 
    |-- organic         -->  794 


In [16]:
# TASK 2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    f"{dataset_folder}/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_data = val_datagen.flow_from_directory(
    f"{dataset_folder}/validation",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Found 2002 images belonging to 3 classes.
Found 525 images belonging to 3 classes.


In [17]:
# TASK 3

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    
    Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128,activation='relu'),
    Dropout(0.5),

    Dense(3,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,347 (42.61 MB)

 Trainable params: 11,169,347 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 247s 4s/step - accuracy: 0.6853 - loss: 0.7067 - val_accuracy: 0.7105 - val_loss: 0.6773
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 243s 4s/step - accuracy: 0.6843 - loss: 0.6840 - val_accuracy: 0.6781 - val_loss: 0.6047
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 252s 4s/step - accuracy: 0.7008 - loss: 0.6660 - val_accuracy: 0.7048 - val_loss: 0.6212
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 244s 4s/step - accuracy: 0.7048 - loss: 0.6447 - val_accuracy: 0.6781 - val_loss: 0.6781
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 241s 4s/step - accuracy: 0.7023 - loss: 0.6412 - val_accuracy: 0.6514 - val_loss: 0.6512


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

In [ ]:
# TASK 4

from sklearn.metrics import confusion_matrix
import numpy as np

predictions = model.predict(val_data)

y_pred = np.argmax(predictions,axis=1)
y_true = val_data.classes

cm = confusion_matrix(y_true,y_pred)

print(cm)

In [ ]:
import seaborn as sns

sns.heatmap(cm,annot=True,fmt="d")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true,y_pred)
print("Accuracy:",acc)

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img("test/plastic.jpg",target_size=(224,224))

img_array = image.img_to_array(img)/255
img_array = np.expand_dims(img_array,axis=0)

prediction = model.predict(img_array)

classes = ["recyclable","organic","non_recyclable"]

print("Prediction:",classes[np.argmax(prediction)])

In [ ]:
# TASK 5

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128,activation='relu')(x)

output = Dense(3,activation='softmax')(x)

model_tl = Model(inputs=base_model.input,outputs=output)

model_tl.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_tl = model_tl.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)